### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [1]:
# Install pinned dependencies (Colab-ready; safe to re-run).
# Based on latest compatible versions
!pip install -q langchain-core==1.6.3 langchain-openai==1.6.2 python-dotenv==1.2.3 tiktoken==0.14.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 10.3 MB/s eta 0:00:00


## Tutorial: Building a Q&A LLMChain
We’ll build a minimal question-answering chain over provided context.

Learning outcomes:
- Craft a Q&A prompt that cites context
- Run a chain with inputs: `question`, `context`
- Add guardrails: answer only from context, else say “I don’t know”


In [2]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter OPENROUTER_API_KEY (hidden): ")
MODEL = "openai/gpt-4o-mini"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", temperature=0, seed=42)


Enter OPENROUTER_API_KEY (hidden): ··········


### Step 1: Create Q&A prompt
Keep it extractive and cite source spans when possible.


In [4]:
qa_template = (
    "You are a terse and accurate assistant. Use the CONTEXT to answer the QUESTION.\n"
    "If the answer is not in the CONTEXT, say: I don't know.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)
qa_prompt = PromptTemplate.from_template(qa_template)
qa_chain = qa_prompt | llm | StrOutputParser()


### Step 2: Run the chain on sample context
We simulate a small knowledge base paragraph.


In [6]:
context = (
    "LangChain is a framework for building applications with LLMs. "
    "It provides abstractions like PromptTemplate, Chains, Tools, and Agents, "
    "making it easier to compose multi-step workflows."
)

print(qa_chain.invoke({
    "question": "What does LangChain help developers do?",
    "context": context
}))

print(qa_chain.invoke({
    "question": "What training data does LangChain use?",
    "context": context
}))


LangChain helps developers build applications with LLMs by providing abstractions for composing multi-step workflows.
I don't know.
